# Pruning the BirdCLEF 2020

We will prune the BirdCLEF 2020 dataset to according to MetaAudio's approach. To make training faster, we will select only a subset of this dataset for pre-training and training. MetaAudio's approach involves removing audios...

- More than 180 seconds long
- Belonging to subrepresented classes (less than 50 samples)

In our case, we will extend this approach. We consider...

- Audios at most 60 seconds long
- 5% most represented classes for train and test (random 70%/30% split for each class)
- 5% to 25% most represented classes for unlabeled pretraining

### Extracting valid samples

We save all paths valid samples (less than 60 seconds, including header) to a .json file so we can process them later

In [ ]:
# Folders
INPUT_FOLDER = "../data/BirdCLEF"
OUTPUT_FOLDER = "../data/PrunedBirdCLEF"

# Criteria for discarding
MAX_LENGTH = 60

# Split
LABELED_SPLIT = 0.05
UNLABELED_SPLIT = 0.25
TRAIN_TEST_SPLIT = 0.7

# Audio compression
BITRATE = "96k"

In [3]:
import os
from mutagen.mp3 import MP3
from tqdm.auto import tqdm
import json


if not os.path.isdir(OUTPUT_FOLDER):
    os.mkdir(OUTPUT_FOLDER)

total_sample_count = 0
valid_sample_count = 0
data = []
discarded_files = {
    "too_long": [],
    "cant_load": []
}
for folder_name in tqdm(os.listdir(INPUT_FOLDER), desc="Scanning folders", unit=" classes"):
    folder_path = INPUT_FOLDER + "/" + folder_name
    if not os.path.isdir(folder_path):
        continue
    
    info = {
        "name": folder_name,
        "files": []
    }
    for file_name in os.listdir(folder_path):
        total_sample_count += 1

        file_path = folder_path + "/" + file_name

        # Remove non .mp3
        if not os.path.isfile(file_path) or not file_path.endswith(".mp3"): # Remove non .mp3
            continue

        # Remove missing header
        try: 
            mp3 = MP3(file_path)
        except:
            discarded_files["cant_load"].append(file_path)
            continue

        # Remove too long
        if mp3.info.length > MAX_LENGTH:
            discarded_files["too_long"].append(file_path)
            continue
        
        # Add files
        valid_sample_count += 1
        info["files"].append(file_path)
    data.append(info)

# Save results
with open(f"{OUTPUT_FOLDER}/valid_inputs.json", "w") as f:
    json.dump({"info": data, "discarded": discarded_files, "initial_sample_count": total_sample_count, "current_sample_count": valid_sample_count}, f, indent=4)

# Print data
print(f"Reduced sample count from {total_sample_count} to {valid_sample_count} ({(total_sample_count - valid_sample_count) / total_sample_count * 100:.1f}% decrease)")

Scanning folders:   0%|          | 0/960 [00:00<?, ? classes/s]

Reduced sample count from 72307 to 56892 (21.3% decrease)


### Separating labeled and unlabeled data

We separate labeled and unlabeled data based on representativeness. We sort classes in terms of sample count and select:

- Top 10% for labeled
- Top 10% to 50% for unlabeled
- Bottom 50% discarded

In [ ]:
import json
import subprocess
from tqdm.auto import tqdm

with open(f"{OUTPUT_FOLDER}/valid_inputs.json", "r") as f:
    input_info = json.load(f)

data = sorted(input_info["info"], key=lambda d: len(d["files"]), reverse=True)

dataset_specs = {
    "train": [],
    "test": [],
    "unlabeled": [],
    "label_names": {}
}

if not os.path.isdir(f"{OUTPUT_FOLDER}/audio"):
    os.mkdir(f"{OUTPUT_FOLDER}/audio")

def add_to_split(input_path, split, label=-1):
    # Get output path
    split_input_path = input_path.split("/")
    class_folder = f"{OUTPUT_FOLDER}/audio/{split_input_path[-2]}"
    if not os.path.isdir(class_folder):
        os.mkdir(class_folder)
    output_path = f"{class_folder}/{split_input_path[-1]}"
    local_output_path = f"/audio/{split_input_path[-2]}/{split_input_path[-1]}"

    if not os.path.isfile(output_path):
        # Compress
        try:
            command = ["ffmpeg", "-i", input_path, "-b:a", BITRATE, output_path]
            subprocess.run(command,
                        stdout=subprocess.DEVNULL, 
                        stderr=subprocess.STDOUT)
        except:
            print(f"Couldn't compress sound file @ {input_path}")
            return

    # Add to dataset
    dataset_specs[split].append((local_output_path, label))

# Labeled split
for i in tqdm(range(0, int(LABELED_SPLIT * len(data))), "Compressing labeled split", unit=" classes"):
    # Add to train
    for j in range(0, int(TRAIN_TEST_SPLIT * len(data[i]["files"]))):
        add_to_split(data[i]["files"][j], "train", i)
    # Add to test
    for j in range(int(TRAIN_TEST_SPLIT * len(data[i]["files"])), len(data[i]["files"])):
        add_to_split(data[i]["files"][j], "test", i)
    
    dataset_specs["label_names"][i] = data[i]["files"][0].split("/")[-2]

# Unlabeled split
for i in tqdm(range(int(LABELED_SPLIT * len(data)), int(UNLABELED_SPLIT * len(data))), "Compressing unlabeled split", unit=" classes"):
    for j in range(0, len(data[i]["files"])):
        add_to_split(data[i]["files"][j], "unlabeled")

# Save results
with open(f"{OUTPUT_FOLDER}/dataset_specs.json", "w") as f:
    json.dump(dataset_specs, f, indent=4)

# Print results
print(f"Train set - {len(dataset_specs['train'])} samples; {int(LABELED_SPLIT * len(data))} classes")
print(f"Test set - {len(dataset_specs['test'])} samples; {int(LABELED_SPLIT * len(data))} classes")
print(f"Unlabeled set - {len(dataset_specs['unlabeled'])} samples; {int(UNLABELED_SPLIT * len(data)) - int(LABELED_SPLIT * len(data))} classes")

current_sample_count = len(dataset_specs["train"]) + len(dataset_specs["test"]) + len(dataset_specs["unlabeled"])
print(f"Reduced from {input_info['initial_sample_count']} to {current_sample_count} ({(input_info['initial_sample_count'] - current_sample_count) / input_info['initial_sample_count'] * 100:.1f}% decrease)")

Compressing labeled split:   0%|          | 0/48 [00:00<?, ? classes/s]

Compressing unlabeled split:   0%|          | 0/192 [00:00<?, ? classes/s]

Train set - 3090 samples; 48 classes
Test set - 1361 samples; 48 classes
Unlabeled set - 16188 samples; 192 classes
Reduced from 72307 to 20639 (71.5% decrease)


### Removing unused files

We do a final pass to remove previously compressed files in the output folder that aren't effectively used in the final dataset

In [5]:
import os
import json
import subprocess

with open(f"{OUTPUT_FOLDER}/dataset_specs.json", "r") as f:
    specs = json.load(f)

for dir_path, dir_names, file_names in tqdm(os.walk(f"{OUTPUT_FOLDER}/audio/"), "Searching for unused classes", unit=" classes"):
    for file_name in file_names:
        file_path = os.path.join(dir_path, file_name)
        
        found = False
        for split in ["train", "test", "unlabeled"]:
            for entry in specs[split]:
                if entry[0] == file_path:
                    found = True
                    break
            if found:
                break
        if not found:
            os.remove(file_path)
    
    contents = os.listdir(dir_path)
    if len(contents) == 0:
        os.rmdir(dir_path)



Searching for unused classes: 0 classes [00:00, ? classes/s]

### Calculating size reduction

We compared the size of the original dataset with our pruned and compressed one

In [6]:
import os

def get_dir_size(start_path = '.'):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(start_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            # skip if it is symbolic link
            if not os.path.islink(fp):
                total_size += os.path.getsize(fp)

    return total_size / (1024**3)

og = get_dir_size(INPUT_FOLDER)
new = get_dir_size(OUTPUT_FOLDER)

print(f"Reduced dataset from {og:.2f}GB to {new:.2f}GB ({(og - new) / og * 100:.2f}% reduction)")

Reduced dataset from 63.64GB to 5.44GB (91.45% reduction)


### Creating a DataLoader

Now we create a DataLoader from our processed dataset to make training easier

In [7]:
import wave, sys
import numpy as np
import matplotlib.pyplot as plt

def visualize(ax, waveform, label):
    """
    Plots the waveform with the label on the given ax
    """

    # Reads all frames
    signal = waveform.readframes(-1)
    signal = np.frombuffer(signal, dtype="int16")

    # Gets frame rate
    f_rate = waveform.getframerate()

    time = np.linspace(
        0,
        len(signal) / f_rate,
        num = len(signal)
    )

    ax.set_title(label)